In [ ]:
!pip install -U sentence-transformers
!pip install rank_bm25
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.6/340.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 17.5 MB/s eta 0:00:00


In [ ]:
import requests
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from rank_bm25 import BM25Okapi

In [55]:
# Updated API Endpoint for Semantic Scholar
SEMANTIC_SCHOLAR_BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search"
HEADERS = {
    "User-Agent": "DeepCite/1.0",
}

# Function to fetch papers from Semantic Scholar API based on a query
def fetch_papers_by_keywords(keywords, fields="title,abstract,url,year,citationCount,authors", limit=1000):
    papers = []
    offset = 0
    batch_size = 100  # Smaller batch size to avoid rate limiting

    while len(papers) < limit:
        params = {
            "query": keywords,
            "fields": fields,
            "limit": min(batch_size, limit - len(papers)),
            "offset": offset
        }

        try:
            response = requests.get(SEMANTIC_SCHOLAR_BASE_URL, headers=HEADERS, params=params)

            if response.status_code == 200:
                data = response.json()
                new_papers = data.get("data", [])

                # Break if no new papers or fewer papers than requested (end of results)
                if not new_papers:
                    break

                papers.extend(new_papers)
                offset += len(new_papers)

                # Add delay between requests to avoid rate limiting
                time.sleep(1)

            elif response.status_code == 429:
                print(f"Rate limit exceeded. Waiting 30 seconds before retry...")
                time.sleep(30)  # Wait longer if rate limited
                continue
            else:
                print(f"Failed to fetch data, status code: {response.status_code}")
                break

        except Exception as e:
            print(f"Error fetching papers: {str(e)}")
            break

        # Check if we've reached all available papers
        if len(papers) >= offset:
            break

    return papers[:limit]

In [56]:
# Initialize the SentenceTransformer model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

In [57]:
def get_embeddings(papers):
    """Generate embeddings for the list of papers using SentenceTransformer."""
    if not papers:
        return None

    titles_and_abstracts = []
    for paper in papers:
        title = paper.get('title', '')  # Default to empty string if title is missing
        abstract = paper.get('abstract', '')  # Default to empty string if abstract is missing
        title = title if title else ''  # Ensure title is a string
        abstract = abstract if abstract else ''  # Ensure abstract is a string
        titles_and_abstracts.append(title + " " + abstract)

    embeddings = model.encode(titles_and_abstracts, convert_to_tensor=True)
    return embeddings

In [58]:
# Function to create and store embeddings in FAISS index
def create_faiss_index(embeddings):
    """Create and store embeddings in FAISS index."""
    if embeddings is None:
        return None

    # Convert embeddings to a NumPy array for FAISS
    embeddings_np = np.array(embeddings.cpu().detach().numpy()).astype('float32')

    # Create the FAISS index (using L2 distance, Euclidean)
    index = faiss.IndexFlatL2(embeddings_np.shape[1])
    index.add(embeddings_np)  # Add embeddings to the index
    return index

In [59]:
def get_bm25_scores(query, papers):
    """Compute BM25 similarity scores between query and papers."""
    if not papers:
        return []

    # Preprocess documents: tokenize title + abstract
    tokenized_corpus = [
        ((paper.get('title') or '') + ' ' + (paper.get('abstract') or '')).lower().split()
        for paper in papers]
    bm25 = BM25Okapi(tokenized_corpus)

    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    return scores

In [60]:
def search_query(query, faiss_index, papers, top_k=20):
    """Search query in FAISS index and retrieve semantic + BM25 scores."""
    if not papers or faiss_index is None:
        return [], [], []

    # Limit top_k to the number of available papers
    top_k = min(top_k, len(papers))
    if top_k == 0:
        return [], [], []

    # FAISS
    query_embedding = model.encode([query], convert_to_tensor=True)
    query_embedding_np = np.array(query_embedding.cpu().detach().numpy()).astype('float32')
    distances, indices = faiss_index.search(query_embedding_np, top_k)

    # Make sure indices are unique to avoid duplicates
    unique_indices = []
    seen = set()
    for idx in indices[0]:
        if idx not in seen and idx < len(papers):
            seen.add(idx)
            unique_indices.append(idx)

    recommended_papers = [papers[i] for i in unique_indices]

    # Get corresponding distances for the unique indices
    faiss_dists = [distances[0][list(indices[0]).index(i)] for i in unique_indices]

    # BM25
    bm25_scores = get_bm25_scores(query, recommended_papers)

    return recommended_papers, faiss_dists, bm25_scores

In [71]:
def rank_by_weighted_score_hybrid(papers, faiss_dists, bm25_scores, weights=(0.5, 0.3, 0.2), hybrid_weights=(0.7, 0.3)):
    """Rank papers with hybrid relevance (FAISS + BM25) and citation/recency."""
    relevance_weight, citation_weight, recency_weight = weights
    w_faiss, w_bm25 = hybrid_weights

    # Normalize FAISS distances into similarity
    max_dist = max(faiss_dists) + 1e-5
    faiss_sims = [1 - (d / max_dist) for d in faiss_dists]

    # Normalize BM25 scores
    max_bm25 = max(bm25_scores) + 1e-5
    bm25_sims = [s / max_bm25 for s in bm25_scores]

    # Normalize citation and recency
    max_citation = max(p['citationCount'] for p in papers) + 1e-5
    max_year = max(p['year'] for p in papers) + 1e-5
    min_year = min(p['year'] for p in papers)

    ranked = []
    for i, paper in enumerate(papers):
        combined_relevance = (w_faiss * faiss_sims[i]) + (w_bm25 * bm25_sims[i])
        norm_citation = paper['citationCount'] / max_citation
        norm_recency = (paper['year'] - min_year) / (max_year - min_year)

        final_score = (
            relevance_weight * combined_relevance +
            citation_weight * norm_citation +
            recency_weight * norm_recency
        )

        ranked.append({
            "Title": paper['title'],
            "Abstract": paper.get('abstract', 'N/A'),
            "DOI": paper.get('url', "N/A"),
            "Citation Count": paper['citationCount'],
            "Year": paper['year'],
            "FAISS Similarity": round(faiss_sims[i], 4),
            "BM25 Similarity": round(bm25_sims[i], 4),
            "Combined Relevance": round(combined_relevance, 4),
            "Final Score": round(final_score, 4)
        })

    ranked.sort(key=lambda x: -x['Final Score'])
    return ranked

In [72]:
# Main function to fetch, rank, and return relevant papers based on a query
def get_ranked_papers(query, weights=(0.5, 0.3, 0.2), hybrid_weights=(0.7, 0.3), limit=100):
    print(f"Searching for papers on: '{query}'")
    papers = fetch_papers_by_keywords(query, limit=limit)

    if not papers:
        return "No relevant papers found."

    print(f"Found {len(papers)} papers. Creating embeddings...")

    embeddings = get_embeddings(papers)
    faiss_index = create_faiss_index(embeddings)

    print("Searching and ranking papers...")
    recommended_papers, faiss_dists, bm25_scores = search_query(query, faiss_index, papers, top_k=min(20, len(papers)))

    if not recommended_papers:
        return "No relevant papers could be ranked."

    ranked_papers = rank_by_weighted_score_hybrid(recommended_papers, faiss_dists, bm25_scores, weights, hybrid_weights)
    return ranked_papers

In [73]:
# Example usage
def run_query(query, limit=100):
    ranked_papers = get_ranked_papers(query, limit=limit)

    if isinstance(ranked_papers, str):
        print(ranked_papers)
        return

    # Output the ranked papers in a clean user-friendly format
# Output the ranked papers in a clean user-friendly format
    print("\nTop Recommended Papers:\n")
    for i, paper in enumerate(ranked_papers[:10], 1):
        print(f"{i}. {paper['Title']}")
        print(f"   Relevance Score: {round(paper['Final Score'] * 100, 2)}%")
        print(f"   Abstract: {paper['Abstract'] or 'N/A'}")
        print(f"   DOI: {paper['DOI']}")
        print(f"   Citations: {paper['Citation Count']} | Year: {paper['Year']}")
        print()


    return ranked_papers

In [74]:
# If you want to run this directly:
if __name__ == "__main__":
    query = "Apple Silicon Architecture"
    # Lower the limit to avoid rate limiting during testing
    run_query(query, limit=100)

Searching for papers on: 'Apple Silicon Architecture'
Found 100 papers. Creating embeddings...
Searching and ranking papers...

Top Recommended Papers:

1. Profiling Apple Silicon Performance for ML Training
   Relevance Score: 41.79%
   Abstract: Apple Silicon has attracted much attention for its performance and role in machine learning (ML) training. Unlike NVIDIA GPUs, which have traditionally dominated ML training, Apple Silicon has a significant difference in memory architecture. It uses Unified Memory, which integrates CPU and GPU memory instead of separate CPU memory and GPU VRAM. However, it is difficult to tell whether Unified Memory means more performance benefits. This paper investigates the performance differences by training several large language model (LLM) workloads end-to-end under different memory scenarios. The results show a significant performance gap between Apple Silicon and NVIDIA GPUs. This paper attributes this gap to system-level factors such as page faults, 